<a href="https://colab.research.google.com/github/amurock3-web/UI_Detection_model_training/blob/master/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UI detector — full training run (Colab T4)

Trains the 16-class UI element detector. Produces `best.pt` plus the run
artifacts, zipped and downloaded to your machine.

**Run the cells in order.** Every cell that can fail, fails loudly and says
what to do — nothing here degrades quietly.

### Before you start

1. **Runtime > Change runtime type > T4 GPU.** Cell 1 stops if you skipped this.
2. **This repo needs a git remote.** It has none yet. From your machine:
   ```
   gh repo create UI_Detection_model_training --private --source=. --push
   ```
   then paste the URL into `REPO_URL` in cell 2.
3. **Have the dataset ready.** It is client data and is *never* in git — see
   the dataset cell below for exactly where to put it.

### What this costs

83 training images at `imgsz=1280`, `batch=8` is ~11 batches/epoch. On a T4
expect roughly **1.5-2.5 hours** for 200 epochs, often less because
`patience=50` stops early. That fits inside a normal Colab session, but Colab
can still kill you without warning — which is why the training cell zips and
downloads in the *same* cell, and why the Drive cell below is worth doing.

In [1]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "\n" + "!" * 70 +
        "\nNO GPU. Runtime > Change runtime type > T4 GPU, then Run all again."
        "\nTraining on CPU would take days, so this stops here.\n" + "!" * 70)

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {name}")
print(f"VRAM   : {vram:.1f} GB")
print(f"torch  : {torch.__version__}")

# config/train.yaml (imgsz 1280, batch 8) is tuned for a 16GB T4.
if vram < 14:
    print("\n" + "!" * 70)
    print(f"WARNING: {vram:.1f} GB is below the 16 GB this config assumes.")
    print("If training OOMs, step down in this order (CLAUDE.md):")
    print("  1. batch: 8 -> 4     2. imgsz: 1280 -> 640     3. freeze: 10")
    print("!" * 70)

Thu Sep 10 15:34:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo

`REPO_URL` is empty on purpose — this repo has no remote yet. Create one and
paste the URL here. A private repo is correct: while the *code* is safe to
push, keeping it private means an accidental `git add data/` is not instantly
public.

In [2]:
# ── 2. Clone ────────────────────────────────────────────────────────────────
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/amurock3-web/UI_Detection_model_training"            # <-- paste your remote here
BRANCH   = "master"
REPO_DIR = Path("/content/UI_Detection_model_training")

if not REPO_URL:
    raise SystemExit(
        "\n" + "!" * 70 +
        "\nREPO_URL is empty. This repo has no git remote yet. On your machine:"
        "\n    gh repo create UI_Detection_model_training --private --source=. --push"
        "\nthen paste the URL above and rerun this cell."
        "\n\nNo remote and no intention of making one? Zip the repo (WITHOUT"
        "\ndata/), upload it with the Files pane, and unzip to the path above."
        "\n" + "!" * 70)

if REPO_DIR.exists():
    print(f"{REPO_DIR} already present - pulling instead of cloning")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("\ncwd:", Path.cwd())
print("sha:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True).stdout.strip())

/content/UI_Detection_model_training already present - pulling instead of cloning

cwd: /content/UI_Detection_model_training
sha: 0a6e1d2


In [3]:
# ── 3. Dependencies ─────────────────────────────────────────────────────────
# Colab ships a CUDA torch that already satisfies requirements.txt, so pip
# leaves it alone. If pip *does* replace torch, restart the runtime and rerun.
%pip install -q -r requirements.txt

import ultralytics, torch
ultralytics.checks()
print("torch CUDA still available:", torch.cuda.is_available())

Ultralytics 8.4.146 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 48.8/112.6 GB disk)
torch CUDA still available: True


## 4. Supply the dataset — read this carefully

The 103 labelled screenshots are **client data**. They are not in git and never
will be (`.gitignore` blocks `data/`). You supply them here, one of two ways.

What the notebook needs is the **original renamed dataset**, not the split:

```
data/renamed/
├── images/          ss001.png … ss103.png     (103 files)
├── labels/          ss001.txt … ss103.txt     (103 files, ORIGINAL 22-class)
└── classes_22.txt   22 lines
```

The 16-class labels and the train/val split are **regenerated here** rather
than uploaded. That is deliberate: `config/ui.yaml` is written with an absolute
path by `split_dataset.py`, so the one in git points at a Windows drive and is
meaningless on Colab. Regenerating also proves the Colab split is byte-identical
to your local one — the cell after this asserts exactly that.

**Option A — Google Drive (recommended).** Put `renamed.zip` in your Drive at
`MyDrive/ui16_data/renamed.zip`, then run the cell with `USE_DRIVE = True`.

**Option B — direct upload.** Set `USE_DRIVE = False` and pick `renamed.zip`
when prompted. Slower, and lost when the session dies.

Zip it on your machine from the repo root:
```
cd data && zip -r renamed.zip renamed/
```

In [4]:
# ── 4. Dataset ──────────────────────────────────────────────────────────────
import shutil, zipfile
from pathlib import Path

USE_DRIVE = True
DRIVE_ZIP = "/content/drive/MyDrive/ui16_data/renamed.zip"

DATA = Path("data")
RENAMED = DATA / "renamed"
DATA.mkdir(exist_ok=True)

if not RENAMED.exists():
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        src = Path(DRIVE_ZIP)
        if not src.exists():
            raise SystemExit(
                "\n" + "!" * 70 +
                f"\nNot found: {src}"
                "\nPut renamed.zip there, or set USE_DRIVE = False to upload it."
                "\n" + "!" * 70)
    else:
        from google.colab import files
        print("Pick renamed.zip ...")
        uploaded = files.upload()
        src = Path(next(iter(uploaded)))
    # Zips built by Windows PowerShell store "renamed\images\ss001.png" with
    # backslashes. Linux treats those as one long filename, not folders, so
    # extractall() silently yields a flat pile of junk. Normalise as we go.
    with zipfile.ZipFile(src) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if name.endswith("/"):
                continue
            target = DATA / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as a, open(target, "wb") as b:
                shutil.copyfileobj(a, b)

# Fail loudly and specifically - a half-present dataset must not reach training.
problems = []
n_img = len(list((RENAMED / "images").glob("*.png"))) if (RENAMED / "images").exists() else 0
n_lbl = len(list((RENAMED / "labels").glob("*.txt"))) if (RENAMED / "labels").exists() else 0
if not RENAMED.exists():
    problems.append(f"{RENAMED} does not exist")
if n_img != 103:
    problems.append(f"{RENAMED}/images has {n_img} png, expected 103")
if n_lbl != 103:
    problems.append(f"{RENAMED}/labels has {n_lbl} txt, expected 103")
if not (RENAMED / "classes_22.txt").exists():
    problems.append(f"{RENAMED}/classes_22.txt missing")

if problems:
    raise SystemExit("\n" + "!" * 70 + "\nDATASET NOT USABLE:\n  - " +
                     "\n  - ".join(problems) +
                     "\n\nExpected layout:\n"
                     "  data/renamed/images/ssNNN.png      (103)\n"
                     "  data/renamed/labels/ssNNN.txt      (103, original 22-class)\n"
                     "  data/renamed/classes_22.txt        (22 lines)\n" + "!" * 70)

print(f"OK: {n_img} images, {n_lbl} labels, classes_22.txt present")

OK: 103 images, 103 labels, classes_22.txt present


In [5]:
# ── 5. Remap 22 -> 16, then split ───────────────────────────────────────────
# Both are deterministic (seed 42, Mersenne Twister), so this reproduces the
# exact split from the local machine. The assert below proves it.
!python scripts/remap_labels.py
!python scripts/split_dataset.py

from pathlib import Path

EXPECTED_VAL = {"ss002", "ss011", "ss012", "ss017", "ss030", "ss032", "ss036",
                "ss038", "ss047", "ss049", "ss050", "ss066", "ss073", "ss074",
                "ss075", "ss082", "ss085", "ss089", "ss091", "ss092"}
actual_val = {p.stem for p in Path("ds/images/val").iterdir()}

if actual_val != EXPECTED_VAL:
    raise SystemExit(
        "\n" + "!" * 70 +
        "\nSPLIT DRIFTED from the local run - results will not be comparable."
        f"\n  only here  : {sorted(actual_val - EXPECTED_VAL)}"
        f"\n  only local : {sorted(EXPECTED_VAL - actual_val)}"
        "\nDo not train on this. Check the dataset and the seed first.\n" + "!" * 70)

print(f"\nSplit matches the local run exactly: {len(actual_val)} val images")
print(Path("config/ui.yaml").read_text()[:220])

REMAP 22 -> 16
label files : 103
instances   : 2186 -> 2186

new id  new class        count   = sum of originals
--------------------------------------------------------------
     0  content_card       657   = content_card 657   then +20 from corrections
     1  tab_item           458   = carousel_filter 26 + category_tab 85 + detail_tab 34 + nav_item 313   then +10 from corrections
     2  overlay_badge      302   = badge 264 + duration 27 + episode_badge 11   then +4 from corrections
     3  section_title      193   = section_title 193   then -21 from corrections
     4  button             105   = button 105   then -14 from corrections
     5  cast_card           95   = cast_card 95
     6  icon                94   = icon 94   then +1 from corrections
     7  metadata_text       88   = metadata_text 88
     8  navigation_bar      71   = navigation_bar 71
     9  celebrity_card      47   = celebrity_card 47
    10  title               28   = movie_title 24 + series_title 4
    11  de

## 6. Point `runs/` at Drive (optional but recommended)

Ultralytics rewrites `best.pt` every time the model improves. If `runs/` lives
in Drive, each improvement is already saved off-machine — so a session kill at
epoch 180 costs you nothing instead of everything.

Skip this if you did not mount Drive. The training cell still zips and
downloads either way.

In [6]:
# ── 6. runs/ -> Drive (optional) ────────────────────────────────────────────
import os
from pathlib import Path

DRIVE_RUNS = Path("/content/drive/MyDrive/ui16_runs")

if Path("/content/drive/MyDrive").exists():
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
    if not Path("runs").exists():
        os.symlink(DRIVE_RUNS, "runs")
    print(f"runs/ -> {DRIVE_RUNS}  (weights survive a session kill)")
else:
    print("Drive not mounted - runs/ stays local. The training cell still")
    print("downloads the zip, but an early session kill loses the run.")

runs/ -> /content/drive/MyDrive/ui16_runs  (weights survive a session kill)


## 7. Train, then immediately zip and download

**One cell on purpose.** Colab kills sessions without warning; if the download
lived in a later cell you could lose a two-hour run to a disconnect between
cells.

The zip and download run in a `finally`, so even a crashed or interrupted run
still hands back whatever artifacts exist.

Settings come from `config/train.yaml` — `yolov8s.pt`, `imgsz=1280`,
`epochs=200`, `batch=8`, `patience=50`, plus the augmentation overrides
(`fliplr=0.0`, `hsv_h=0.0`, `mosaic=0.3`, …) that exist because UI screenshots
are not photographs. `scripts/train.py` also writes `run_meta.json` recording
the git sha, split seed and per-class mAP.

In [ ]:
# ── 7. TRAIN + ZIP + DOWNLOAD (do not split this cell) ──────────────────────
import shutil, subprocess, time
from datetime import datetime
from pathlib import Path

started = time.time()
rc = None
try:
    # config/train.yaml supplies yolov8s.pt / 1280 / 200 / batch 8 / patience 50
    rc = subprocess.call(["python", "scripts/train.py"])
    print(f"\ntrain.py exit code: {rc}")
finally:
    mins = (time.time() - started) / 60
    print(f"elapsed: {mins:.1f} min")

    if Path("runs").exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        archive = shutil.make_archive(f"/content/ui16_runs_{stamp}", "zip", "runs")
        size = Path(archive).stat().st_size / 1e6
        print(f"zipped: {archive}  ({size:.1f} MB)")

        weights = sorted(Path("runs").rglob("best.pt"))
        try:
            from google.colab import files
            files.download(archive)
            for w in weights:                      # named so two runs cannot collide
                tagged = Path(f"/content/best_{w.parent.parent.name}_{stamp}.pt")
                shutil.copy2(w, tagged)
                files.download(str(tagged))
            print(f"downloads started: 1 zip + {len(weights)} best.pt")
        except Exception as exc:                   # non-Colab, or popup blocked
            print(f"auto-download failed ({exc}). Grab it from the Files pane: {archive}")

        for w in weights:
            print("  best.pt:", w, f"{w.stat().st_size / 1e6:.1f} MB")
    else:
        print("no runs/ directory - training never started")

if rc != 0:
    raise SystemExit(f"training failed (exit {rc}) - artifacts above were still saved")

## 8. Alternative architecture — `yolo11s.pt`

For the Phase 10 architecture sweep. Uncomment and run *after* the baseline
above, so both runs land in the same `runs/` and the comparison is like for
like. Everything else is held constant — same split, same seed, same
augmentation — so any difference is the architecture.

In [ ]:
# ── 8. yolo11s baseline comparison (commented out by design) ────────────────
# Uncomment the block below to run it. Same data, same seed, same augmentation.
#
# import shutil, subprocess, time
# from datetime import datetime
# from pathlib import Path
#
# started = time.time()
# rc = None
# try:
#     rc = subprocess.call(["python", "scripts/train.py",
#                           "--model", "yolo11s.pt", "--name", "ui16_yolo11s"])
#     print(f"\ntrain.py exit code: {rc}")
# finally:
#     print(f"elapsed: {(time.time() - started) / 60:.1f} min")
#     if Path("runs").exists():
#         stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#         archive = shutil.make_archive(f"/content/ui16_yolo11s_{stamp}", "zip", "runs")
#         print("zipped:", archive)
#         try:
#             from google.colab import files
#             files.download(archive)
#         except Exception as exc:
#             print(f"auto-download failed ({exc}) - use the Files pane: {archive}")
#
# # Compare the two runs:
# # import json
# # for meta in sorted(Path("runs").rglob("run_meta.json")):
# #     m = json.loads(meta.read_text())
# #     print(f"{m['run']:<18}{m['model']:<14}mAP50-95 {m['metrics']['map50_95']}")